# ReAct + Reflexion Agent: Ollama 8B + LangChain/LangGraph + Streamlit + Cloudflare Tunnel
논문 **「ReAct: Synergizing Reasoning and Acting in Language Models (ICLR 2023)」** 및 **「Reflexion: Language Agents with Verbal Reinforcement Learning (2023)」** 이론을 OpenAI API Key 없이 **Colab 로컬 Ollama(Llama 3.1 8B)**로 구현하고, **Streamlit 웹 앱 + Cloudflare Tunnel**로 즉시 시연할 수 있는 통합 노트북입니다.

### 실행 순서
위에서부터 차례로 실행하면 됩니다: ① 패키지 설치 → ② Ollama 서버·모델 → ③ `agent_core.py` 생성 → ④ LangChain/LangGraph 테스트(선택) → ⑤ `app.py` 생성 → ⑥ Streamlit + 터널 시작.
코드를 고친 뒤에는 ③·⑤·⑥ 셀만 다시 실행하면 앱이 새 코드로 재시작됩니다.

### v3 수정 사항 (환각 이름·오답 대응)
1. **지어낸 한글 이름 차단**: `search` 인자의 한글 단어가 질문·Observation 어디에도 없으면 실행하지 않고 거절 ('페버스' 같은 음차 환각).
2. **동음이의 문서 자동 해소**: `search[이민진]` → 질문의 '작가'로 `[이민진 (작가)]`, `search[파친코]` → 도박 문서 대신 소설/드라마 문서. 문서 머리에 영어 제목(`en: ...`)도 표시.
3. **근거 인용 검증**: `finish`에 `evidence`(Observation 원문 인용)를 요구하고 실제로 Observation에 있는지 문자열로 확인. 없으면 2번까지 거절.
4. **속성 혼동 방지**: '한국계 미국인(국적)'을 '출생지'로 답하는 식의 오답을 Actor·Evaluator 프롬프트에서 명시적으로 차단.
5. **lookup 개선**: 현재 문서 제목을 알려주고, 앞서 받은 web_search 결과에서도 찾음. 다단어 키워드도 부분 일치.
6. **웹 결과 정리**: HTML 엔티티 복원, 네이버 UI 잔여물('클립', '더보기') 제거, 질의 관련도 순 상위 8개만 전달.
7. **Evaluator 모델 분리**: Reflexion 사이드바에서 채점·성찰 모델을 따로 고를 수 있음 (8B 자기채점은 관대함).

### v2 수정 사항
1. **Wikipedia `Expecting value: line 1 column 1 (char 0)` 오류 해결**: `wikipedia` 패키지의 기본 User-Agent가 Wikimedia에서 차단되어 JSON이 아닌 응답이 오던 문제. MediaWiki API를 정책 준수 User-Agent로 직접 호출하도록 교체 (재시도·상태 코드 확인 포함).
2. **같은 행동 무한 반복 차단**: 중복 Action은 실행하지 않고, Wikipedia 장애 시 같은 스텝에서 `web_search`로 자동 폴백. 진전 없는 스텝이 연속되면 조기 종료.
3. **스텝 한도 도달 시**: 확보된 Observation만으로 답을 종합 (근거가 없을 때만 실패 메시지).
4. **JSON 모드 + `num_ctx=8192`**: 8B 모델의 JSON 깨짐과 긴 궤적에서의 컨텍스트 잘림 방지.
5. **공용 코어 분리**: 노트북 테스트 셀과 Streamlit 앱이 `agent_core.py` 하나를 공유 (중복 구현 제거).
6. **HITL 사고 편집 UI 실제 구현**, 실시간 스텝 표시, 대화창 스트리밍·역할 구분 수정.

> Wikipedia 요청 한도를 더 받으려면 ③ 셀 실행 전에 `import os; os.environ["WIKI_CONTACT"] = "본인 메일 또는 URL"` 을 설정하세요.

In [ ]:
# 1) Colab Ubuntu 환경에서 Ollama 설치에 필요한 유틸리티 설치
!apt-get -qq update && apt-get -qq install -y zstd pciutils curl

# 2) Colab 사전 설치 패키지(langgraph-prebuilt 1.x 등)와 버전 충돌이 없도록 함께 동기화 업그레이드
#    (wikipedia / langchain-community 는 더 이상 사용하지 않음 — MediaWiki API 직접 호출로 교체)
!pip install -q --upgrade "langgraph>=1.0.0" "langgraph-prebuilt>=1.0.0" "langchain>=1.0.0" "langchain-classic>=1.0.0" "langchain-ollama" "langchain-core" "ddgs" "beautifulsoup4" "streamlit"

import importlib
importlib.invalidate_caches()
print("✅ 모든 Python 패키지 동기화 설치 완료 (이미 이전 셀을 실행했던 세션이라면 [런타임 > 세션 다시 시작] 후 다음 셀부터 실행하세요)")

In [ ]:
import os
import shutil
import subprocess
import time
import requests

MODEL_NAME = "llama3.1:8b"
# (선택) 한국어 질문·Evaluator용으로 함께 받아둘 모델. 예: ["qwen3:8b"] — 앱 사이드바에서 선택 가능
EXTRA_MODELS = []
OLLAMA_URL = "http://127.0.0.1:11434"

# 1) Ollama 설치 (이미 설치되어 있으면 건너뜀)
if shutil.which("ollama") is None:
    print("⏳ Ollama 설치 중...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# 2) 기존 프로세스 정리 후 백그라운드 실행 (로그는 ollama.log 에 남겨 문제 발생 시 확인 가능)
subprocess.run(["pkill", "-f", "ollama serve"], check=False)
time.sleep(1)
env = dict(os.environ, OLLAMA_KEEP_ALIVE="30m")   # 모델을 VRAM에 유지 → 매 요청마다 재로딩 방지
subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, env=env)

# 3) /api/tags 헬스체크 폴링 (최대 30초 대기)
for _ in range(30):
    try:
        if requests.get(f"{OLLAMA_URL}/api/tags", timeout=1).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama 서버가 시작되지 않았습니다. ollama.log 를 확인하세요.")
print("✅ Ollama 서버 정상 가동 확인 (127.0.0.1:11434)")

# 4) Tool Calling을 네이티브 지원하는 Llama 3.1 (8B) 모델 다운로드 (이미 있으면 건너뜀)
def installed_models():
    r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    r.raise_for_status()
    return [m["name"] for m in r.json().get("models", [])]

if not any(MODEL_NAME in m for m in installed_models()):
    print(f"⏳ {MODEL_NAME} 모델 다운로드 중 (최초 1회 약 4.9GB)...")
    subprocess.run(["ollama", "pull", MODEL_NAME], check=True)
for extra in EXTRA_MODELS:
    if not any(extra in m for m in installed_models()):
        print(f"⏳ 추가 모델 {extra} 다운로드 중...")
        subprocess.run(["ollama", "pull", extra], check=True)
if not any(MODEL_NAME in m for m in installed_models()):
    raise RuntimeError(f"{MODEL_NAME} 다운로드 확인 실패. 현재 모델: {installed_models()}")

# 5) 워밍업: 앱과 동일한 num_ctx(8192)로 미리 VRAM에 올려 첫 질문의 로딩 지연 제거
t0 = time.time()
requests.post(
    f"{OLLAMA_URL}/api/generate",
    json={"model": MODEL_NAME, "prompt": "hi", "stream": False, "keep_alive": "30m",
          "options": {"num_ctx": 8192, "num_predict": 1}},
    timeout=300,
).raise_for_status()
print(f"✅ 준비 완료된 Ollama 모델: {installed_models()} (워밍업 {time.time() - t0:.1f}s)")

In [ ]:
%%writefile agent_core.py
"""agent_core.py — ReAct / Reflexion / RAG 공용 코어 (Streamlit 비의존).

app.py(Streamlit UI)와 노트북 테스트 셀이 함께 임포트해서 쓰는 단일 구현입니다.
- Wikipedia: `wikipedia` 패키지 대신 MediaWiki API를 직접 호출 (정책 준수 User-Agent + 재시도)
- Web search: 엔진 병렬 호출 + 전체 타임아웃 + TTL 캐시
- ReAct loop: 중복 행동 차단, Wikipedia 장애 시 web_search 자동 폴백, 스텝 한도 도달 시 관찰 기반 종합
"""
from __future__ import annotations

import ast
import html
import json
import operator
import os
import re
import time
import urllib.parse
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor, wait
from difflib import SequenceMatcher
from functools import lru_cache
from math import log

import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from langchain_core.messages import HumanMessage, SystemMessage

try:
    from ddgs import DDGS
except ImportError:  # 구 패키지명 폴백
    from duckduckgo_search import DDGS

OLLAMA_URL = "http://127.0.0.1:11434"
MAX_STEP_FAIL_MSG = "최대 스텝 내에 검증된 확정 답변을 생성하지 못했습니다."
OBS_LIMIT = 1200          # Observation 1건당 최대 글자 수 (8B 컨텍스트 보호)
HIST_OBS_LIMIT = 600      # 프롬프트 히스토리에 다시 넣을 때의 Observation 길이

# ---------------------------------------------------------
# 0. HTTP 세션
# ---------------------------------------------------------
# Wikimedia User-Agent 정책: 연락처(메일/URL)가 포함된 고유 UA가 아니면 403/429 + 비-JSON 본문을 돌려준다.
# (`wikipedia` 패키지의 기본 UA가 바로 이 경우 → "Expecting value: line 1 column 1 (char 0)")
# 본인 메일/URL을 환경변수 WIKI_CONTACT 에 넣으면 더 높은 rate limit 버킷을 받는다.
WIKI_UA = (
    "ReActReflexionColabLab/2.0 "
    f"({os.environ.get('WIKI_CONTACT', 'https://colab.research.google.com; educational ReAct demo')}) "
    f"python-requests/{requests.__version__}"
)
BROWSER_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def _make_session(headers: dict) -> requests.Session:
    s = requests.Session()
    retry = Retry(
        total=3, backoff_factor=0.6,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",), respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=8)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update(headers)
    return s


_WIKI = _make_session({"User-Agent": WIKI_UA, "Accept": "application/json"})
_WEB = _make_session(BROWSER_HEADERS)

_HANGUL = re.compile(r"[가-힣]")


def _is_ko(text: str) -> bool:
    return bool(_HANGUL.search(text))


def _detect_wiki_lang(text: str, default_lang: str = "en") -> str:
    return "ko" if (default_lang == "ko" or _is_ko(text)) else "en"


# ---------------------------------------------------------
# 1. Safe calculator
# ---------------------------------------------------------
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod,
    ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _safe_eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        left, right = _safe_eval_node(node.left), _safe_eval_node(node.right)
        if isinstance(node.op, ast.Pow) and (abs(right) > 100 or abs(left) > 1e6):
            raise ValueError("Exponent too large")
        return _OPS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval_node(node.operand))
    raise ValueError("Unsupported expression")


def safe_calc(expression: str) -> str:
    """'2,884 + 220' 같은 순수 산술식을 안전하게 계산한다."""
    expr = expression.strip().strip("\"'").replace(",", "").replace("×", "*").replace("÷", "/").replace("^", "**")
    val = _safe_eval_node(ast.parse(expr, mode="eval").body)
    if isinstance(val, float) and val.is_integer():
        val = int(val)
    return str(val)


# ---------------------------------------------------------
# 2. Wikipedia (MediaWiki API 직접 호출)
# ---------------------------------------------------------
class WikiError(RuntimeError):
    pass


def _wiki_get(lang: str, **params) -> dict:
    params = {"format": "json", "formatversion": 2, **params}
    try:
        r = _WIKI.get(f"https://{lang}.wikipedia.org/w/api.php", params=params, timeout=8)
    except requests.RequestException as exc:
        raise WikiError(f"network error ({type(exc).__name__})") from exc
    if r.status_code != 200:
        raise WikiError(f"HTTP {r.status_code} ({r.text[:80].strip()!r})")
    try:
        data = r.json()
    except ValueError as exc:
        raise WikiError(f"non-JSON response ({r.text[:80].strip()!r})") from exc
    if "error" in data:
        raise WikiError(str(data["error"].get("info", data["error"])))
    return data


_STOP = frozenset(
    "the a an of on in for and or to is was are were who which what whom whose by with from at as be did does do "
    "that this it its".split()
)
_QUAL = re.compile(r"^(.*?)\s*\(([^()]*)\)\s*$")


def _ctx_features(text: str) -> frozenset[str]:
    return frozenset(f for f in _features(text) if f not in _STOP) if text else frozenset()


@lru_cache(maxsize=512)
def _wiki_page(title: str, lang: str) -> dict | None:
    data = _wiki_get(
        lang, action="query", prop="extracts|pageprops|langlinks", explaintext=1, exsectionformat="plain",
        redirects=1, ppprop="disambiguation", lllang="en" if lang == "ko" else "ko", titles=title,
    )
    pages = data.get("query", {}).get("pages", [])
    if not pages or pages[0].get("missing") or pages[0].get("invalid"):
        return None
    return pages[0]


@lru_cache(maxsize=512)
def _wiki_titles(query: str, lang: str, limit: int = 8) -> tuple[str, ...]:
    data = _wiki_get(lang, action="query", list="search", srsearch=query, srlimit=limit, srprop="")
    return tuple(h["title"] for h in data.get("query", {}).get("search", []))


@lru_cache(maxsize=256)
def _wiki_disambig_links(title: str, lang: str) -> tuple[str, ...]:
    data = _wiki_get(lang, action="query", prop="links", plnamespace=0, pllimit=100, titles=title)
    pages = data.get("query", {}).get("pages", [])
    return tuple(link["title"] for link in (pages[0].get("links", []) if pages else []))


def clear_caches() -> None:
    for fn in (_wiki_page, _wiki_titles, _wiki_disambig_links):
        fn.cache_clear()
    _WEB_CACHE.clear()


def _qual_score(title: str, ctx: frozenset[str]) -> int:
    """'파친코 (소설)'의 괄호 안 한정어가 질문 키워드와 몇 개 겹치는지. 한정어가 없으면 0."""
    m = _QUAL.match(title)
    return len(_ctx_features(m.group(2)) & ctx) if m else 0


def _best_option(options, ctx: frozenset[str], floor: int = 0) -> str | None:
    best, best_score = None, floor
    for t in options:
        sc = _qual_score(t, ctx)
        if sc > best_score:
            best, best_score = t, sc
    return best


def _split_sentences(text: str) -> list[str]:
    out = []
    for para in text.split("\n"):
        para = para.strip()
        if len(para) < 25:      # 섹션 제목/빈 줄 제거
            continue
        out.extend(s.strip() for s in re.split(r"(?<=[.!?])\s+", para) if s.strip())
    return out[:600]


def wiki_search_api(entity: str, lang: str = "en", context: str = "") -> tuple[str, list[str]]:
    """Wikipedia에서 entity 문서를 찾아 (observation, 문장 리스트)를 반환.

    context(질문 등)가 주어지면 동음이의 문서를 질문 키워드로 자동 해소한다.
    예) search[이민진] + 질문에 '작가' → [이민진 (작가)],  search[파친코] + 질문에 '소설' → [파친코 (소설)]
    """
    clean = re.sub(r"\s+", " ", entity.strip().strip("\"'[]"))
    if not clean:
        return "Empty search query. Provide an entity title.", []
    eff = _detect_wiki_lang(clean, lang)
    langs = [eff, "en"] if eff == "ko" else [eff]
    ctx = _ctx_features(context)
    try:
        for lg in langs:
            note = ""
            page = _wiki_page(clean, lg)
            if page is None:
                titles = _wiki_titles(clean, lg)
                if not titles:
                    continue
                page = _wiki_page(titles[0], lg)
                if page is None:
                    continue
                note = f"(No exact page for [{clean}]; loaded top search hit. Other candidates: {list(titles[1:5])}) "
            title = page.get("title", clean)
            m = _QUAL.match(title)
            base = m.group(1) if m else title

            if "disambiguation" in page.get("pageprops", {}):
                links = _wiki_disambig_links(title, lg)
                options = [t for t in links if t.startswith(base) and t != title] or list(links)
                pick = _best_option(options, ctx)
                target = _wiki_page(pick, lg) if pick else None
                if target is None or "disambiguation" in target.get("pageprops", {}):
                    return f"Disambiguation for [{title}] ({lg}.wiki). Search one of these exact titles: {options[:8]}", []
                others = [o for o in options if o != pick][:5]
                note += f"(Disambiguation [{title}] → auto-selected [{target['title']}] by question keywords. Other options: {others}) "
                page = target
            elif ctx:
                # 동명 문서 중 질문 맥락에 더 맞는 '제목 (한정어)' 문서가 있으면 그쪽을 연다 (실패해도 원래 문서 유지)
                try:
                    variants = [t for t in _wiki_titles(base, lg) if t != title and t.startswith(base + " (")]
                    pick = _best_option(variants, ctx, floor=_qual_score(title, ctx))
                    target = _wiki_page(pick, lg) if pick else None
                    if target and target.get("extract") and "disambiguation" not in target.get("pageprops", {}):
                        note += f"([{title}] is a different topic from the question; loaded [{target['title']}] instead.) "
                        page = target
                except WikiError:
                    pass

            sentences = _split_sentences(page.get("extract", ""))
            if not sentences:
                continue
            ll = page.get("langlinks") or []
            alt = f" · {ll[0].get('lang')}: {ll[0].get('title')}" if ll and ll[0].get("title") else ""
            obs = f"{note}[{page.get('title', clean)} ({lg}.wiki){alt}] " + " ".join(sentences[:5])
            return obs[:OBS_LIMIT], sentences
    except Exception as exc:  # WikiError 포함 — 예외는 lru_cache에 저장되지 않으므로 다음 호출에서 재시도된다
        return (
            f"Wikipedia lookup error: {exc}. Do NOT repeat this same action; "
            "use web_search or a different exact title.",
            [],
        )
    return (
        f"Could not find [{clean}] on Wikipedia. Use a shorter exact entity title "
        "(e.g. a person/work name, not a full question) or call web_search.",
        [],
    )


def _match_sentences(keyword: str, pool: list[str]) -> list[str]:
    key = keyword.strip().strip("\"'").lower()
    exact = [x for x in pool if key in x.lower()]
    if exact:
        return exact
    words = [w for w in _TOKEN.findall(key) if w not in _STOP]
    if len(words) < 2:
        return []
    need = max(2, -(-len(words) * 2 // 3))      # 다단어 키워드: 단어의 2/3 이상 포함한 문장
    scored = [(sum(w in x.lower() for w in words), i, x) for i, x in enumerate(pool)]
    return [x for sc, _, x in sorted(scored, key=lambda t: (-t[0], t[1])) if sc >= need]


def wiki_lookup_api(keyword: str, sentences: list[str], max_hits: int = 3,
                    page_title: str = "", web_snippets: list[str] | None = None) -> str:
    """현재 Wikipedia 문서(없으면 앞서 받은 web_search 결과)에서 키워드가 든 문장을 찾는다."""
    if not keyword.strip():
        return "Empty lookup keyword."
    if not sentences and not web_snippets:
        return "No Wikipedia page loaded yet. Call search[entity] first."
    where = f"page [{page_title}]" if page_title else "current page"
    hits = _match_sentences(keyword, sentences)
    if hits:
        return f"({where} · Result 1-{min(max_hits, len(hits))} / {len(hits)}) {' ... '.join(hits[:max_hits])}"[:OBS_LIMIT]
    hits = _match_sentences(keyword, web_snippets or [])
    if hits:
        return f"(Not in {where}; found in earlier web_search results) {' ... '.join(hits[:max_hits])}"[:OBS_LIMIT]
    return (f"No sentence containing '{keyword}' in {where} or earlier web results. "
            "Use ONE short keyword (e.g. 'born', '태어'), or search another entity.")


# ---------------------------------------------------------
# 3. 병렬 멀티엔진 웹 검색
# ---------------------------------------------------------
def _ddgs_text(q: str, max_results: int, backend: str) -> list[dict]:
    region = "kr-kr" if _is_ko(q) else "wt-wt"
    try:
        client = DDGS(timeout=6)
    except TypeError:
        client = DDGS()
    try:
        return list(client.text(q, region=region, max_results=max_results, backend=backend))
    except TypeError:
        return list(client.text(q, max_results=max_results))


def _fmt_hits(tag: str, hits: list[dict]) -> list[str]:
    out = []
    for r in hits:
        title = str(r.get("title", "")).strip()
        body = str(r.get("body", "")).strip()[:200]
        if title or body:
            out.append(f"[{tag}] {title}: {body}")
    return out


def _search_duckduckgo(q: str, max_results: int = 3) -> list[str]:
    try:
        return _fmt_hits("DuckDuckGo", _ddgs_text(q, max_results, "duckduckgo"))
    except Exception:
        return []


def _search_google(q: str, max_results: int = 3) -> list[str]:
    for backend in ("google", "bing", "brave"):
        try:
            out = _fmt_hits("Google" if backend == "google" else f"Google/{backend.title()}",
                            _ddgs_text(q, max_results, backend))
            if out:
                return out
        except Exception:
            continue
    # 모든 백엔드가 막힌 경우에만 Google News RSS 폴백 (뉴스 제목은 사실 질의에 잡음이 되므로 항상 붙이지 않는다)
    try:
        hl, gl, ceid = ("ko", "KR", "KR:ko") if _is_ko(q) else ("en-US", "US", "US:en")
        resp = _WEB.get(
            "https://news.google.com/rss/search",
            params={"q": q, "hl": hl, "gl": gl, "ceid": ceid}, timeout=6,
        )
        if resp.status_code == 200:
            root = ET.fromstring(resp.content)
            return [
                f"[Google News] {it.findtext('title', '').strip()}"
                for it in root.findall(".//item")[:max_results] if it.findtext("title")
            ]
    except Exception:
        pass
    return []


def _naver_encyc(encoded: str) -> list[str]:
    out = []
    try:
        r = _WEB.get(f"https://terms.naver.com/search.naver?query={encoded}", timeout=6)
        if r.status_code == 200:
            soup = BeautifulSoup(r.text, "html.parser")
            for li in soup.select("ul.content_list li")[:2]:
                t_el = li.select_one(".title a") or li.select_one(".title")
                d_el = li.select_one("p.desc") or li.select_one(".desc")
                title = t_el.get_text(" ", strip=True) if t_el else ""
                desc = d_el.get_text(" ", strip=True)[:200] if d_el else ""
                if title or desc:
                    out.append(f"[Naver 지식백과] {title}: {desc}")
    except Exception:
        pass
    return out


def _naver_web(encoded: str, max_results: int) -> list[str]:
    out: list[str] = []
    seen: set[str] = set()
    try:
        r = _WEB.get(f"https://search.naver.com/search.naver?where=nexearch&query={encoded}", timeout=6)
        if r.status_code != 200:
            return out
        soup = BeautifulSoup(r.text, "html.parser")
        for t_sel, d_sel in ((".total_tit", ".total_dsc"), (".news_tit", ".news_dsc"),
                             ("a.link_tit", ".api_txt_lines"), (".question_text", ".answer_text")):
            descs = soup.select(d_sel)
            for idx, t_el in enumerate(soup.select(t_sel)[:max_results]):
                d_txt = descs[idx].get_text(" ", strip=True)[:200] if idx < len(descs) else ""
                combined = f"{t_el.get_text(' ', strip=True)}: {d_txt}".strip(": ")
                if combined and combined not in seen:
                    seen.add(combined)
                    out.append(f"[Naver] {combined}")
            if len(out) >= max_results:
                break
        if len(out) < 2:
            for p in soup.select(".api_txt_lines.dsc_txt, .sds-comps-text-type-body1")[:3]:
                txt = p.get_text(" ", strip=True)[:200]
                if txt and txt not in seen:
                    seen.add(txt)
                    out.append(f"[Naver] {txt}")
    except Exception:
        pass
    return out


def _search_naver(q: str, max_results: int = 3) -> list[str]:
    encoded = urllib.parse.quote_plus(q)
    with ThreadPoolExecutor(max_workers=2) as ex:      # 지식백과 + 통합검색을 동시에
        f1, f2 = ex.submit(_naver_encyc, encoded), ex.submit(_naver_web, encoded, max_results)
        return (f1.result() + f2.result())[:max_results + 1]


def _search_wiki_fulltext(q: str, max_results: int = 3) -> list[str]:
    try:
        data = _wiki_get("ko" if _is_ko(q) else "en", action="query", list="search",
                         srsearch=q, srlimit=max_results)
        return [
            f"[WikiFullText:{h.get('title', '')}] " + re.sub(r"<[^>]+>", "", h.get("snippet", "")).strip()
            for h in data.get("query", {}).get("search", [])
        ]
    except Exception:
        return []


ENGINE_MAP = {
    "DuckDuckGo": _search_duckduckgo,
    "Google": _search_google,
    "Naver": _search_naver,
    "Wikipedia 본문": _search_wiki_fulltext,
}
ALL_ENGINES = list(ENGINE_MAP)
NO_WEB_MSG = "No web results found."
MAX_WEB_ITEMS = 8

_WEB_CACHE: dict[tuple, tuple[float, str]] = {}
_WEB_CACHE_TTL = 600


def web_search_api(query: str, engines: list[str] | None = None, max_per_engine: int = 3,
                   total_timeout: float = 12.0) -> str:
    clean_q = re.sub(r"\s+", " ", query.strip().strip("\"'[]"))
    if not clean_q:
        return NO_WEB_MSG
    active = [e for e in (engines or ALL_ENGINES) if e in ENGINE_MAP]
    key = (clean_q.lower(), tuple(active), max_per_engine)
    hit = _WEB_CACHE.get(key)
    if hit and time.time() - hit[0] < _WEB_CACHE_TTL:
        return hit[1]

    # 느린 엔진 하나가 전체를 붙잡지 않도록 전체 타임아웃을 건다 (늦은 엔진 결과는 버림)
    executor = ThreadPoolExecutor(max_workers=max(1, len(active)))
    futures = {executor.submit(ENGINE_MAP[name], clean_q, max_per_engine): name for name in active}
    done, _ = wait(futures, timeout=total_timeout)
    executor.shutdown(wait=False, cancel_futures=True)
    per_engine = {}
    for f in done:
        try:
            per_engine[futures[f]] = f.result()
        except Exception:
            per_engine[futures[f]] = []

    merged: list[str] = []
    seen: set[str] = set()
    for name in active:                      # 선택한 엔진 순서 유지
        for item in per_engine.get(name, []):
            norm = item.lower().strip()
            if norm and norm not in seen:
                seen.add(norm)
                merged.append(item)
    # 정리: HTML 엔티티 복원, 네이버 UI 잔여물('클립', '더보기' 등) 제거, 질의 관련도 순 재정렬 후 상위만 유지
    cleaned = []
    for item in merged:
        item = html.unescape(item)
        if len(re.sub(r"^\[[^\]]*\]\s*", "", item).strip(" :")) >= 15:
            cleaned.append(item)
    qf = _ctx_features(clean_q)
    ranked = sorted(enumerate(cleaned), key=lambda p: (-len(_features(p[1]) & qf), p[0]))
    if ranked and len(_features(ranked[0][1]) & qf) > 0:
        ranked = [p for p in ranked if len(_features(p[1]) & qf) > 0]
    merged = [it for _, it in ranked[:MAX_WEB_ITEMS]]
    if not merged:
        return NO_WEB_MSG                    # 실패는 캐시하지 않는다
    result = " | ".join(merged)[:OBS_LIMIT + 400]
    _WEB_CACHE[key] = (time.time(), result)
    return result


# ---------------------------------------------------------
# 4. Simple in-memory RAG (단어 + 한글 2-gram, IDF 가중)
# ---------------------------------------------------------
RAG_EMPTY_MSG = "사용자가 업로드한 문서가 없거나 비어 있습니다."
RAG_MISS_MSG = "업로드된 문서에서 질문과 관련된 내용을 찾을 수 없습니다."
_TOKEN = re.compile(r"[가-힣]+|[a-zA-Z0-9]+")


def decode_upload(raw: bytes) -> str:
    for enc in ("utf-8-sig", "cp949", "utf-16"):
        try:
            return raw.decode(enc)
        except UnicodeError:
            continue
    return raw.decode("utf-8", errors="replace")


def chunk_text(text: str, chunk_size: int = 120, overlap: int = 30) -> list[str]:
    words = text.split()
    step = max(1, chunk_size - overlap)
    chunks = []
    for i in range(0, len(words), step):
        chunks.append(" ".join(words[i:i + chunk_size]))
        if i + chunk_size >= len(words):
            break
    return [c for c in chunks if c]


@lru_cache(maxsize=8192)
def _features(text: str) -> frozenset[str]:
    feats = set()
    for tok in _TOKEN.findall(text.lower()):
        feats.add(tok)
        if _HANGUL.match(tok) and len(tok) > 2:      # 조사 변형 대응: '작가는' ↔ '작가가'
            feats.update(tok[i:i + 2] for i in range(len(tok) - 1))
    return frozenset(feats)


def simple_rag_search(query: str, chunks: list[str], top_k: int = 3, chunk_chars: int = 500) -> str:
    if not chunks:
        return RAG_EMPTY_MSG
    q_feats = _features(query)
    chunk_feats = [_features(c) for c in chunks]     # 청크별 토큰 집합은 캐시되어 재계산 없음
    n = len(chunks)
    idf = {f: log(1 + n / df) for f in q_feats if (df := sum(f in cf for cf in chunk_feats))}
    total = sum(idf.values())
    scored = []
    if total:
        for idx, cf in enumerate(chunk_feats):
            score = sum(w for f, w in idf.items() if f in cf) / total
            if score > 0:
                scored.append((score, idx))
    scored.sort(key=lambda x: (-x[0], x[1]))
    if scored:
        picked = [idx for _, idx in scored[:top_k]]
        head = ""
    else:
        # '핵심 내용을 요약해줘'처럼 키워드가 문서에 없는 질의 → 문서 앞부분을 근거로 제공
        picked = list(range(min(top_k, n)))
        head = "(키워드 일치 없음 — 문서 앞부분을 제공합니다)\n\n"
    return head + "\n\n".join(f"[업로드 문서 발췌 {i + 1}/{n}] {chunks[i][:chunk_chars]}" for i in picked)


# ---------------------------------------------------------
# 5. LLM helpers
# ---------------------------------------------------------
def make_llm(model_name: str, temperature: float = 0.0, num_predict: int = 512, num_ctx: int = 8192):
    """ChatOllama 생성. num_ctx를 명시하지 않으면 긴 궤적이 기본 컨텍스트에서 잘려 앞부분(시스템 프롬프트)이 유실된다."""
    from langchain_ollama import ChatOllama
    return ChatOllama(
        model=model_name, base_url=OLLAMA_URL, temperature=temperature,
        num_ctx=num_ctx, num_predict=num_predict, keep_alive="30m",
    )


def _extract_json(raw: str) -> dict | None:
    m = re.search(r"\{.*\}", raw, re.S)
    for cand in ([m.group(0)] if m else []) + [raw]:
        try:
            obj = json.loads(cand)
            if isinstance(obj, dict):
                return obj
        except ValueError:
            continue
    return None


def _invoke_json(llm, messages) -> tuple[dict | None, str]:
    """Ollama JSON 모드(format='json')로 호출 → 8B 모델의 JSON 깨짐을 원천 차단."""
    try:
        raw = llm.invoke(messages, format="json").content
    except TypeError:
        raw = llm.invoke(messages).content
    return _extract_json(raw), raw


_ACTION_RE = re.compile(r"^\s*([a-z_ ]+?)\s*\[(.*)\]\s*$", re.I | re.S)
_ALIASES = {
    "wikipedia": "search", "wiki": "search", "search_wikipedia": "search", "wiki_search": "search",
    "websearch": "web_search", "web": "web_search", "google": "web_search", "internet_search": "web_search",
    "rag": "rag_retriever", "retriever": "rag_retriever", "retrieve": "rag_retriever",
    "calculate": "calc", "calculator": "calc", "math": "calc",
    "answer": "finish", "final": "finish", "final_answer": "finish", "done": "finish",
}


def _normalize_action(parsed: dict) -> tuple[str, str]:
    act_type = str(parsed.get("action_type", "")).strip()
    act_arg = parsed.get("action_arg", "")
    if isinstance(act_arg, (list, tuple)):
        act_arg = " ".join(map(str, act_arg))
    act_arg = str(act_arg).strip()
    m = _ACTION_RE.match(act_type)           # 모델이 action_type에 "search[...]" 통째로 넣는 경우
    if m:
        act_type, act_arg = m.group(1).lower().strip(), act_arg or m.group(2).strip()
    act_type = act_type.lower().strip().replace(" ", "_")
    return _ALIASES.get(act_type, act_type), act_arg


def _act_key(act_type: str, act_arg: str) -> str:
    return f"{act_type}[{re.sub(r'[^0-9a-z가-힣]+', ' ', act_arg.lower()).strip()}]"


_FAIL_MARKERS = ("Wikipedia lookup error", "Could not find", "Disambiguation for", NO_WEB_MSG,
                 RAG_MISS_MSG, RAG_EMPTY_MSG, "Calc error", "Duplicate action", "Unknown action",
                 "Finish rejected", "Rejected search", "is disabled", "No Wikipedia page loaded", "No sentence containing",
                 "Empty ", "Invalid JSON")


_PARTICLE = re.compile(r"(으로|에서|에게|은|는|이|가|을|를|의|에|와|과|로|도)$")


def _invented_hangul(arg: str, context: str) -> list[str]:
    """search 인자의 한글 단어가 질문·Observation 어디에도 없으면(=모델이 지어낸 음차) 그 단어들을 반환."""
    toks = re.findall(r"[가-힣]{2,}", arg)
    if not toks:
        return []
    for t in toks:
        stem = _PARTICLE.sub("", t)
        if t in context or (len(stem) >= 2 and stem in context):
            return []
    return toks


def _norm(x: str) -> str:
    return re.sub(r"\s+", " ", html.unescape(str(x))).strip().lower()


def evidence_supported(evidence: str, observations: list[str]) -> bool:
    """finish의 근거 인용이 실제 Observation 안에 (거의) 그대로 있는지 문자열로 검증."""
    e = _norm(evidence).strip("\"'“”‘’ .…")
    if len(e) < 4:
        return False
    blob = _norm(" ".join(observations))
    if e in blob:
        return True
    if len(e) < 12:
        return False
    m = SequenceMatcher(None, blob, e, autojunk=False).find_longest_match(0, len(blob), 0, len(e))
    return m.size >= max(12, int(0.7 * len(e)))


ATTRIBUTE_RULE = ("Answer exactly the attribute the question asks. A related but different attribute is WRONG: "
                  "birthplace ≠ nationality/ethnicity (e.g. 'Korean-American') ≠ residence; "
                  "author ≠ director; publication year ≠ broadcast year.")


def _page_title(obs: str) -> str:
    m = re.search(r"\[([^\[\]]+?) \((?:ko|en)\.wiki", obs)
    return m.group(1) if m else ""


def _is_grounded_obs(action: str, observation: str) -> bool:
    return (
        action.lower().startswith(("search[", "lookup[", "web_search[", "rag_retriever[", "calc["))
        and not str(observation).startswith(_FAIL_MARKERS)
    )


# ---------------------------------------------------------
# 6. ReAct loop (Actor M_a)
# ---------------------------------------------------------
def run_react_loop(
    llm,
    question: str,
    lang: str = "en",
    max_steps: int = 7,
    enable_web: bool = False,
    engines: list[str] | None = None,
    preset_history: list[dict] | None = None,
    rag_chunks: list | None = None,
    episodic_memory: list[str] | None = None,
    on_step=None,
):
    """Thought → Action → Observation 루프. (steps, answer) 반환.

    preset_history의 마지막 항목이 thought만 있고 action이 없으면(HITL 편집) 그 Thought를 그대로 따라 이어서 실행한다.
    on_step(step_dict)은 스텝이 확정될 때마다 호출된다(실시간 UI 표시용).
    """
    steps = [dict(s) for s in preset_history] if preset_history else []
    page_sentences: list[str] = []
    page_title = ""
    web_snippets: list[str] = []
    executed: dict[str, str] = {}
    has_verified_obs = False

    # 이전 궤적 복원 (Wikipedia 페이지는 캐시에서 다시 로드되므로 네트워크 비용 없음)
    for prev in steps:
        m = _ACTION_RE.match(prev.get("action", "") or "")
        if not m:
            continue
        a_type, a_arg = m.group(1).lower(), m.group(2)
        executed[_act_key(a_type, a_arg)] = prev.get("observation", "")
        if a_type == "search":
            obs0, sents = wiki_search_api(a_arg, lang=lang, context=question)
            if sents:
                page_sentences, page_title = sents, _page_title(obs0)
        elif a_type == "web_search":
            web_snippets.extend(str(prev.get("observation", "")).split(" | "))
        if _is_grounded_obs(prev.get("action", ""), prev.get("observation", "")):
            has_verified_obs = True

    target_lang = "Korean" if _detect_wiki_lang(question, lang) == "ko" else "English"
    tool_rules = []
    if rag_chunks:
        tool_rules.append('- "rag_retriever": Search inside the uploaded local document. Use this FIRST when the question refers to the uploaded file. (action_arg = core keywords)')
    tool_rules.append('- "search": Open the Wikipedia page of ONE entity (a person, work, place). action_arg = a short exact title COPIED from the Question or an Observation — never a full question. '
                      'For foreign people/works, prefer the original-language title once an Observation shows it (shown as "en: ..." in the page header). NEVER invent or transliterate a name yourself.')
    tool_rules.append('- "lookup": Find sentences containing ONE short keyword (e.g. "born", "태어") in the opened Wikipedia page and in earlier web_search results.')
    if enable_web:
        tool_rules.append('- "web_search": Multi-engine internet search. Use it when Wikipedia fails or lacks the fact. (action_arg = search query)')
    tool_rules.append('- "calc": Evaluate pure arithmetic like "2884 + 220". (action_arg = math expression)')
    tool_rules.append('- "finish": Give the concise final answer ONLY after an Observation states it. (action_arg = final answer; also add key "evidence" = an exact quote copied from that Observation)')

    memory_block = ""
    if episodic_memory:
        formatted = "\n".join(f"- Past Trial Reflection {i + 1}: {m}" for i, m in enumerate(episodic_memory))
        memory_block = (
            "\n[Long-Term Episodic Memory — Self-Reflections from previous failed trials]\n"
            f"{formatted}\nLearn from these reflections. Do NOT repeat the failed queries.\n"
        )

    sys_prompt = (
        "You are a strictly grounded ReAct Actor (M_a) following Yao et al. (ICLR 2023) and Shinn et al. (Reflexion, 2023).\n"
        'At each step output ONE JSON object with keys "thought", "action_type", "action_arg" (plus "evidence" when finishing).\n'
        "Available action_type values:\n" + "\n".join(tool_rules) + "\n" + memory_block +
        "\nRULES:\n"
        "1. Never answer from memory. Every fact in the final answer must appear in an Observation.\n"
        "2. NEVER repeat an action that already appears in History. If an Observation is an error or lacks the fact, "
        "change the tool or the query.\n"
        "3. Multi-hop questions: resolve one entity per step (e.g. find the author first, then search the author).\n"
        f"4. {ATTRIBUTE_RULE}\n"
        f"5. Write the final answer in {target_lang}.\n"
        "6. Respond ONLY with the JSON object."
    )

    def _emit(step):
        if on_step:
            try:
                on_step(step)
            except Exception:
                pass

    has_pending_edit = bool(steps) and bool(steps[-1].get("thought")) and not steps[-1].get("action")
    start_idx = len(steps) if has_pending_edit else len(steps) + 1
    finish_rejects = 0
    stalled = 0     # 연속으로 진전 없는(중복/형식 오류) 스텝 수

    for step_num in range(start_idx, max_steps + 1):
        is_edited = (step_num == start_idx) and has_pending_edit
        hist_str = "\n".join(
            f"Thought {s['step']}: {s.get('thought', '')}"
            + (f"\nAction {s['step']}: {s['action']}\nObservation {s['step']}: {str(s['observation'])[:HIST_OBS_LIMIT]}"
               if s.get("action") else "")
            for s in steps
        )
        remaining = max_steps - step_num
        if is_edited:
            user_msg = (
                f"Question: {question}\n\nHistory:\n{hist_str}\n\n"
                f"Follow the human-edited Thought {step_num} strictly and output JSON with the action_type and action_arg that carry it out."
            )
        else:
            user_msg = (
                f"Question: {question}\n\nHistory:\n{hist_str or '(Start)'}\n\n"
                f"Generate Step {step_num} JSON. ({remaining} step(s) left after this one"
                + ("; you must finish now if the Observations support an answer" if remaining == 0 else "") + ")"
            )

        parsed, raw = _invoke_json(llm, [SystemMessage(content=sys_prompt), HumanMessage(content=user_msg)])
        if parsed is None:
            # 기존: 파싱 실패 텍스트를 그대로 finish 답변으로 확정 → 수정: 실패로 기록하고 다음 스텝에서 재시도
            thought, act_type, act_arg = raw.strip()[:200], "invalid", ""
            evidence = ""
        else:
            evidence = str(parsed.get("evidence", "") or "").strip()
            thought = str(parsed.get("thought", "")).strip()
            act_type, act_arg = _normalize_action(parsed)
        if is_edited:
            thought = steps[-1]["thought"]

        key = _act_key(act_type, act_arg)
        act_str = f"{act_type}[{act_arg}]"

        # 중복 행동 차단: Wikipedia 검색 반복은 web_search로 자동 전환, 그 외에는 실행하지 않고 경고만 돌려준다
        if key in executed and act_type not in ("finish", "invalid"):
            web_key = _act_key("web_search", act_arg)
            if act_type == "search" and enable_web and web_key not in executed:
                act_type, key = "web_search", web_key
                act_str = f"web_search[{act_arg}]"
                thought += " (동일한 Wikipedia 검색 반복 → web_search로 자동 전환)"
            else:
                obs = ("Duplicate action: this exact action was already executed (see History). "
                       "Choose a different tool or query, or finish if the Observations already contain the answer.")
                act_type = "duplicate"

        grounded_obs = [str(s.get("observation", "")) for s in steps
                        if _is_grounded_obs(s.get("action", ""), s.get("observation", ""))]
        ev_ok = False
        if act_type == "search":
            invented = _invented_hangul(act_arg, question + " " + " ".join(str(s.get("observation", "")) for s in steps))
            if invented:
                act_type = "invented"
                obs = (f"Rejected search: {invented} appears neither in the Question nor in any Observation — "
                       "do not invent or transliterate names. Search an entity written in the Question, "
                       "or a title copied from an Observation.")

        if act_type in ("duplicate", "invented"):
            pass
        elif act_type == "invalid":
            act_str = "invalid[]"
            obs = 'Invalid JSON output. Respond with a single JSON object: {"thought": ..., "action_type": ..., "action_arg": ...}.'
        elif act_type == "rag_retriever":
            obs = simple_rag_search(act_arg, rag_chunks) if rag_chunks else RAG_EMPTY_MSG + " Use another tool."
        elif act_type == "search":
            obs, new_sents = wiki_search_api(act_arg, lang=lang, context=f"{question} {thought}")
            if new_sents:
                page_sentences, page_title = new_sents, _page_title(obs)
            elif obs.startswith("Wikipedia lookup error") and enable_web:
                # Wikipedia 자체가 막힌 경우: 같은 스텝 안에서 웹 검색 결과로 대체해 스텝 낭비를 막는다
                web_obs = web_search_api(act_arg, engines=engines)
                executed[_act_key("web_search", act_arg)] = web_obs
                if web_obs != NO_WEB_MSG:
                    web_snippets.extend(web_obs.split(" | "))
                    obs = f"(Wikipedia unavailable → automatic web_search fallback) {web_obs}"
                    act_str = f"web_search[{act_arg}]"
        elif act_type == "lookup":
            obs = wiki_lookup_api(act_arg, page_sentences, page_title=page_title, web_snippets=web_snippets)
        elif act_type == "web_search":
            obs = web_search_api(act_arg, engines=engines) if enable_web else "web_search is disabled. Use search/lookup instead."
            if obs != NO_WEB_MSG and enable_web:
                web_snippets.extend(obs.split(" | "))
        elif act_type == "calc":
            try:
                obs = f"Result = {safe_calc(act_arg)}"
            except Exception as exc:
                obs = f"Calc error: {exc}"
        elif act_type == "finish":
            ev_ok = evidence_supported(evidence, grounded_obs)
            can_reject = finish_rejects < 2 and step_num < max_steps
            if not act_arg:
                obs, act_type = "Finish rejected: action_arg (the final answer) is empty.", "rejected"
            elif not has_verified_obs and can_reject:
                obs = ("Finish rejected: no Observation supports this answer yet. "
                       "Verify it with search / web_search / rag_retriever first, then finish.")
                act_type = "rejected"
            elif not ev_ok and can_reject:
                obs = ('Finish rejected: "evidence" must be an exact quote copied from an Observation that states the answer. '
                       f"{ATTRIBUTE_RULE} If no Observation states it yet, keep searching (e.g. lookup a keyword like 'born').")
                act_type = "rejected"
            else:
                obs = f"Episode finished. Evidence {'✓ verified' if ev_ok else '✗ unverified'}: {evidence or '(none)'}"
            if act_type == "rejected":
                finish_rejects += 1
        else:
            obs = f"Unknown action_type '{act_type}'. Use one of: search, lookup, web_search, rag_retriever, calc, finish."

        obs = str(obs)[:OBS_LIMIT + 400]
        if act_type not in ("finish", "rejected", "duplicate", "invalid"):
            executed[key] = obs
        if _is_grounded_obs(act_str, obs):
            has_verified_obs = True

        if is_edited:
            steps[-1].update(action=act_str, observation=obs)
        else:
            steps.append({"step": step_num, "thought": thought, "action": act_str, "observation": obs})
        if act_type == "finish":
            steps[-1].update(evidence=evidence, evidence_ok=ev_ok)
        _emit(steps[-1])

        if act_type == "finish":
            return steps, act_arg
        # 모델이 같은 행동에 갇히면 남은 스텝을 태우지 않고 조기 종료 → 아래 종합 단계로 넘어간다
        stalled = stalled + 1 if act_type in ("duplicate", "invalid", "invented") else 0
        if stalled >= 2:
            break

    # 스텝 한도 도달(또는 조기 종료): 확보한 관찰이 있으면 그것만으로 답을 종합한다 (기존: 무조건 실패 메시지)
    if has_verified_obs:
        evidence = "\n".join(
            f"- {s['action']}: {str(s['observation'])[:HIST_OBS_LIMIT]}"
            for s in steps if _is_grounded_obs(s.get("action", ""), s.get("observation", ""))
        )
        synth_prompt = (
            f"Question: {question}\n\nVerified Observations:\n{evidence}\n\n"
            f"Using ONLY these observations, answer in {target_lang}. {ATTRIBUTE_RULE}\n"
            'Respond in JSON: {"asked_attribute": "what the question asks", "evidence": "exact quote copied from an Observation '
            'that states that attribute", "supported": true or false, "answer": "concise answer"}. '
            'Set "supported" to false if no observation states the asked attribute.'
        )
        parsed, _ = _invoke_json(llm, [HumanMessage(content=synth_prompt)])
        quote = str((parsed or {}).get("evidence", "") or "")
        grounded_obs = [str(s["observation"]) for s in steps if _is_grounded_obs(s.get("action", ""), s.get("observation", ""))]
        if (parsed and parsed.get("supported") is True and str(parsed.get("answer", "")).strip()
                and evidence_supported(quote, grounded_obs)):
            answer = str(parsed["answer"]).strip()
            steps.append({
                "step": len(steps) + 1,
                "thought": "(스텝 한도 도달 → 확보된 Observation만으로 답변 종합)",
                "action": f"finish[{answer}]",
                "observation": f"Episode finished (forced synthesis). Evidence ✓ verified: {quote}",
                "evidence": quote, "evidence_ok": True,
            })
            _emit(steps[-1])
            return steps, answer
    return steps, MAX_STEP_FAIL_MSG


# ---------------------------------------------------------
# 7. Reflexion (Shinn et al., 2023 — Algorithm 1)
# ---------------------------------------------------------
def _format_trajectory(steps: list[dict], obs_chars: int = 350) -> str:
    return "\n".join(
        f"Step {s['step']}:\n  Thought: {s.get('thought', '')}\n"
        f"  Action: {s.get('action', '')}\n  Observation: {str(s.get('observation', ''))[:obs_chars]}"
        for s in steps
    )


def evaluate_trajectory(llm, question: str, steps: list[dict], candidate_answer: str) -> dict:
    """Evaluator (M_e): 휴리스틱 궤적 검사 + LLM 이진 평가."""
    flags: list[str] = []
    actions = [s.get("action", "") for s in steps if s.get("action")]
    has_finish = any(a.lower().startswith("finish[") for a in actions)

    if not candidate_answer or MAX_STEP_FAIL_MSG in candidate_answer:
        flags.append("inefficient_planning: 최대 스텝 내에 finish 답변을 완료하지 못함")
    if not has_finish:
        flags.append("incomplete_trajectory: finish[...] 단계 미도달")
    wasted = sum(str(s.get("observation", "")).startswith(("Duplicate action", "Wikipedia lookup error")) for s in steps)
    if wasted >= 3 or any(actions[i] == actions[i + 1] == actions[i + 2] for i in range(len(actions) - 2)):
        flags.append("loop_hallucination: 동일/실패 행동 반복")
    if not any(_is_grounded_obs(s.get("action", ""), s.get("observation", "")) for s in steps):
        flags.append("ungrounded_answer: 외부 검색/RAG 관찰 근거가 확보되지 않음")
    final = next((s for s in reversed(steps) if str(s.get("action", "")).lower().startswith("finish[")), None)
    final_quote = (final or {}).get("evidence", "")
    if final is not None and not final.get("evidence_ok", False):
        flags.append("unverified_evidence: 최종 답변의 근거 인용이 Observation에서 확인되지 않음")

    # 답변 자체가 없으면 LLM 평가를 생략 (어차피 실패 — LLM 호출 1회 절약)
    if not has_finish or MAX_STEP_FAIL_MSG in (candidate_answer or ""):
        return {"passed": False, "score": 0, "reason": "; ".join(flags), "heuristic_flags": flags}

    eval_prompt = f"""You are the Evaluator model (M_e) in the Reflexion framework (Shinn et al., 2023).
Decide whether the candidate answer correctly and completely answers the Question, judged ONLY against the Observations in the trajectory.
{ATTRIBUTE_RULE}

Question: {question}
Candidate Answer: {candidate_answer}
Evidence quoted by the Actor: {final_quote or "(none)"}
Heuristic Flags: {flags or "None"}

Trajectory:
{_format_trajectory(steps)}

Respond strictly in JSON, filling the keys in this order:
{{"asked_attribute": "exactly what the question asks", "evidence_states": "what the evidence actually states",
 "passed": true or false, "score": 1 or 0, "reason": "1-2 sentence diagnostic evaluation"}}"""

    llm_passed, llm_reason = False, "평가 응답 파싱 실패"
    try:
        parsed, _ = _invoke_json(llm, [HumanMessage(content=eval_prompt)])
        if parsed is not None:
            verdict = parsed.get("passed", False)
            llm_passed = verdict if isinstance(verdict, bool) else str(verdict).strip().lower() in ("true", "1", "yes")
            llm_reason = str(parsed.get("reason", "")).strip() or "LLM Evaluator 판정 완료"
    except Exception as exc:
        llm_reason = f"LLM Evaluator 오류: {exc}"

    passed = not flags and llm_passed
    if flags and llm_passed:
        llm_reason = f"휴리스틱 실패({'; '.join(flags)}) — {llm_reason}"
    return {"passed": passed, "score": int(passed), "reason": llm_reason, "heuristic_flags": flags}


def generate_self_reflection(llm, question: str, steps: list[dict], eval_reason: str,
                             episodic_memory: list[str], lang: str = "en") -> str:
    """Self-Reflection (M_sr): 실패 신호를 다음 시행을 위한 언어적 강화 피드백으로 변환."""
    prev_mem = "\n".join(f"- {m}" for m in episodic_memory) if episodic_memory else "(None)"
    reply_lang = "Korean" if _detect_wiki_lang(question, lang) == "ko" else "English"
    sr_prompt = f"""You are the Self-Reflection model (M_sr) in Reflexion (Shinn et al., 2023).
You are given a failed reasoning trajectory, the Evaluator's diagnostic signal, and prior reflections.
Write a concise first-person self-reflection (2-4 sentences in {reply_lang}) that:
1. Pinpoints the exact step or search query that caused the mistake (credit assignment).
2. States a concrete new search entity, keyword, or tool to use in the next trial.
Mention ONLY names that literally appear in the Question or the Observations below — never invent or transliterate names.

Question: {question}
Evaluator Feedback: {eval_reason}
Prior Reflections in Episodic Memory:
{prev_mem}

Failed Trajectory:
{_format_trajectory(steps)}

Self-Reflection:"""
    return llm.invoke([HumanMessage(content=sr_prompt)]).content.strip()


def run_reflexion_loop(
    llm,
    question: str,
    lang: str = "en",
    max_steps: int = 7,
    max_trials: int = 3,
    memory_capacity: int = 3,
    enable_web: bool = False,
    engines: list[str] | None = None,
    rag_chunks: list | None = None,
    on_step=None,
    on_trial=None,
    eval_llm=None,
):
    """Reflexion Algorithm 1: Actor → Evaluator → Self-Reflection 반복.

    eval_llm: Evaluator/Self-Reflection용 모델 (None이면 Actor와 같은 모델). 8B 자기채점은 관대하므로 더 강한 모델 권장.
    """
    critic = eval_llm or llm
    episodic_memory: list[str] = []
    trials: list[dict] = []
    final_answer = ""

    for trial_num in range(1, max_trials + 1):
        if on_trial:
            on_trial(trial_num)
        steps, candidate = run_react_loop(
            llm=llm, question=question, lang=lang, max_steps=max_steps, enable_web=enable_web,
            engines=engines, rag_chunks=rag_chunks, episodic_memory=episodic_memory, on_step=on_step,
        )
        evaluation = evaluate_trajectory(critic, question, steps, candidate)
        final_answer = candidate

        reflection = None
        if not evaluation["passed"]:
            reflection = generate_self_reflection(critic, question, steps, evaluation["reason"], episodic_memory, lang)
            episodic_memory = (episodic_memory + [reflection])[-memory_capacity:]

        trials.append({
            "trial": trial_num, "steps": steps, "answer": candidate, "evaluation": evaluation,
            "reflection": reflection, "memory_snapshot": list(episodic_memory),
        })
        if evaluation["passed"]:
            break

    return trials, final_answer, episodic_memory


In [ ]:
# (선택) LangChain Classic ReAct / LangGraph Tool-Calling 에이전트 동작 테스트
# 도구 구현은 agent_core.py 를 그대로 재사용한다 (앱과 동일한 Wikipedia/검색 코드).
import importlib
import requests
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool

import agent_core as core
importlib.reload(core)   # agent_core.py 를 다시 쓴 경우 최신 내용 반영

# LangChain v1+ (langchain_classic) 및 구버전 (langchain.agents) 모두 호환되도록 Classic ReAct 임포트
try:
    from langchain_classic.agents import create_react_agent as create_classic_react_agent, AgentExecutor
except ImportError:
    from langchain.agents import create_react_agent as create_classic_react_agent, AgentExecutor

# LangChain v1+ 표준 create_agent 우선 사용, 구버전 환경에서는 langgraph.prebuilt 폴백
try:
    from langchain.agents import create_agent as create_modern_agent
except ImportError:
    from langgraph.prebuilt import create_react_agent as create_modern_agent

MODEL_NAME = "llama3.1:8b"
try:
    requests.get(f"{core.OLLAMA_URL}/api/tags", timeout=3).raise_for_status()
except Exception as e:
    raise RuntimeError("Ollama 서버가 꺼져 있습니다. 위의 Ollama 셀(②)을 먼저 실행하세요.") from e

# 0) Wikipedia 연결 자가 진단 — 여기서 'Wikipedia lookup error'가 보이면 네트워크/차단 문제
obs, _ = core.wiki_search_api("Colorado orogeny")
print("▶ Wikipedia 진단:", obs[:200], "\n")

llm = core.make_llm(MODEL_NAME, temperature=0, num_predict=512)

@tool
def search_wikipedia(query: str) -> str:
    """Open the Wikipedia page of one entity and return its first sentences. Pass a short entity title."""
    return core.wiki_search_api(query)[0]

@tool
def safe_calculator(expression: str) -> str:
    """Evaluate a pure mathematical expression safely (e.g., '2884 + 220')."""
    try:
        return core.safe_calc(expression)
    except Exception as e:
        return f"Calculator Error: {e}"

@tool
def web_search(query: str) -> str:
    """Search DuckDuckGo, Google, Naver, and Wikipedia in parallel for verified facts."""
    return core.web_search_api(query)

# 1) Classic ReAct (텍스트 파싱 방식) — hub.pull 네트워크 호출 없이 동일한 프롬프트를 내장
react_prompt = PromptTemplate.from_template(
    "Answer the following questions as best you can. You have access to the following tools:\n\n"
    "{tools}\n\nUse the following format:\n"
    "Question: {input}\nThought: you should always think about what to do\n"
    "Action: the action to take, should be one of [{tool_names}]\n"
    "Action Input: the input to the action\nObservation: the result of the action\n"
    "... (repeat Thought/Action/Action Input/Observation)\n"
    "Thought: I now know the final answer\nFinal Answer: the final answer\n\n"
    "Begin!\n\nQuestion: {input}\nThought:{agent_scratchpad}"
)
classic_tools = [search_wikipedia, safe_calculator]
classic_executor = AgentExecutor(
    agent=create_classic_react_agent(llm=llm, tools=classic_tools, prompt=react_prompt),
    tools=classic_tools, verbose=True, handle_parsing_errors=True, max_iterations=8,
)

q = "What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?"
print("▶ Classic ReAct 테스트:")
try:
    print(classic_executor.invoke({"input": q})["output"])
except Exception as e:
    print(f"Classic ReAct 실행 중 예외 발생: {e}")

# 2) LangGraph Tool-Calling ReAct
graph_agent = create_modern_agent(model=llm, tools=[search_wikipedia, web_search, safe_calculator])
print("\n▶ LangGraph ReAct 테스트:")
try:
    graph_res = graph_agent.invoke({"messages": [HumanMessage(content=q)]}, config={"recursion_limit": 20})
    print(graph_res["messages"][-1].content)
except Exception as e:
    print(f"LangGraph ReAct 실행 중 예외 발생: {e}")

## Streamlit 웹 앱 + Cloudflare Tunnel 배포
아래 셀은 **Streamlit 웹 앱(`app.py`, UI 전용 — 로직은 `agent_core.py`)**을 생성하고, 별도의 계정이나 토큰 없이 **`cloudflared` Quick Tunnel**을 통해 누구나 접속 가능한 실시간 웹 시연 URL(`https://*.trycloudflare.com`)을 발급합니다.

In [ ]:
%%writefile app.py
import requests
import streamlit as st
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool

import agent_core as core

try:
    from langchain.agents import create_agent as create_modern_agent
except ImportError:
    from langgraph.prebuilt import create_react_agent as create_modern_agent

st.set_page_config(
    page_title="ReAct + Reflexion (2023) + RAG · Ollama 8B Live Lab",
    page_icon="⚡",
    layout="wide",
)

# ---------------------------------------------------------
# 1. 캐시된 리소스 (Streamlit은 위젯 조작마다 스크립트 전체를 재실행하므로 무거운 것은 캐시한다)
# ---------------------------------------------------------
@st.cache_data(ttl=15, show_spinner=False)
def check_ollama_models() -> list[str]:
    try:
        r = requests.get(f"{core.OLLAMA_URL}/api/tags", timeout=3)
        if r.status_code == 200:
            return [m["name"] for m in r.json().get("models", [])]
    except Exception:
        pass
    return []


def ensure_ollama_model(model_name: str) -> None:
    models = check_ollama_models()
    if not any(m == model_name or m.startswith(f"{model_name}:") for m in models):
        with st.spinner(f"Ollama 모델 '{model_name}' 다운로드 중 (최초 1회)..."):
            r = requests.post(f"{core.OLLAMA_URL}/api/pull", json={"name": model_name, "stream": False}, timeout=1800)
            r.raise_for_status()
        check_ollama_models.clear()


@st.cache_resource(show_spinner=False)
def get_llm(model_name: str, temperature: float, num_predict: int):
    return core.make_llm(model_name, temperature=temperature, num_predict=num_predict)


@st.cache_data(show_spinner=False)
def build_chunks(raw: bytes) -> list[str]:
    return core.chunk_text(core.decode_upload(raw))


def render_step(s: dict, container=st) -> None:
    if s.get("thought"):
        container.markdown(f"**Thought {s['step']}:** {s['thought']}")
    container.code(f"Action {s['step']}: {s.get('action', '')}\nObservation {s['step']}: {s.get('observation', '')}")


# ---------------------------------------------------------
# 2. Sidebar
# ---------------------------------------------------------
st.title("ReAct + Reflexion + Simple RAG: Verbal RL & Grounded Reasoning")
st.caption("로컬 업로드 문서(RAG) + 실시간 다중 인터넷 검색 + Reflexion(자기성찰 강화학습)이 통합된 Ollama Llama 3.1 8B 데모")

available_models = check_ollama_models()
with st.sidebar:
    st.header("1. Ollama 8B & 에이전트 설정")
    if available_models:
        st.success(f"Ollama 연결됨 ({len(available_models)}개 모델)")
        default_idx = next((i for i, m in enumerate(available_models) if "llama3.1" in m), 0)
        model_name = st.selectbox("모델 선택", available_models, index=default_idx)
    else:
        st.error("Ollama 서버(127.0.0.1:11434)에 연결할 수 없습니다.")
        model_name = st.text_input("모델명", value="llama3.1:8b")

    mode = st.radio(
        "추론·행동 모드",
        [
            "ReAct (Thought → Action → Obs)",
            "Reflexion (Actor → Evaluator → Self-Reflection)",
            "CoT (Reason Only · 비교군)",
            "LangGraph Tool-Calling Agent",
        ],
    )
    wiki_lang = st.selectbox("Wikipedia 언어", ["en", "ko"], index=0, help="질문/검색어에 한글이 있으면 자동으로 ko 위키를 먼저 조회합니다.")
    max_steps = st.slider("최대 추론 스텝", 3, 10, 7)

    max_trials = 3
    memory_capacity = 3
    eval_model_name = model_name
    if mode.startswith("Reflexion"):
        st.subheader("Reflexion 하이퍼파라미터")
        max_trials = st.slider("최대 시행 횟수 (Max Trials)", 1, 5, 3)
        memory_capacity = st.slider("장기 기억 버퍼 크기 (Ω)", 1, 3, 3)
        eval_choices = available_models or [model_name]
        eval_model_name = st.selectbox(
            "Evaluator / Self-Reflection 모델", eval_choices,
            index=eval_choices.index(model_name) if model_name in eval_choices else 0,
            help="같은 8B 모델이 자기 답을 채점하면 관대해집니다. 더 강한 모델을 pull해 두었다면 여기서 고르세요.",
        )

    st.header("2. RAG 지식베이스 설정")
    uploaded_file = st.file_uploader("로컬 텍스트 문서 업로드 (*.txt)", type=["txt"])
    rag_chunks: list[str] = []
    if uploaded_file is not None:
        rag_chunks = build_chunks(uploaded_file.getvalue())
        st.success(f"문서 처리 완료! 총 {len(rag_chunks)}개의 청크 분할됨.")

    st.header("3. 외부 검색망 설정")
    enable_web = st.checkbox("병렬 멀티 검색엔진 활성화", value=True)
    selected_engines = st.multiselect(
        "병렬 검색 엔진 선택", core.ALL_ENGINES, default=core.ALL_ENGINES, disabled=not enable_web,
    )

for key, default in (("react_steps", []), ("react_answer", ""), ("last_question", ""), ("reflexion_trials", []),
                     ("reflexion_memory", []), ("trace_mode", ""), ("run_id", 0), ("chat_history", [])):
    st.session_state.setdefault(key, default)

tab_agent, tab_chat = st.tabs(["🤖 ReAct/Reflexion/RAG 에이전트 분석기", "💬 OLLAMA 8B 실시간 대화창"])

# ---------------------------------------------------------
# 3. Agent tab
# ---------------------------------------------------------
with tab_agent:
    PRESETS = {
        "HotpotQA · Grown-Ups & 'Allo 'Allo! (Reflexion 논문 Fig. 7)": "Grown-Ups starred the actor who was best known for which role on \"'Allo 'Allo!\"?",
        "RAG 전용 테스트 (문서 업로드 후 질문)": "업로드된 문서의 핵심 내용을 바탕으로 핵심 결론을 도출해줘.",
        "HotpotQA · Colorado orogeny (논문 Prompt 예시)": "What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?",
        "한국어 다중 홉 · 파친코 원작 작가": "드라마 파친코의 원작 소설을 쓴 작가는 어느 나라에서 태어났는가?",
    }
    preset_choice = st.selectbox("논문 벤치마크 프리셋 질문", ["직접 입력"] + list(PRESETS.keys()))
    question = st.text_area("질문 입력", value=PRESETS.get(preset_choice, ""), height=80)

    def run_react(preset_history=None):
        """ReAct 루프를 실행하면서 스텝을 실시간으로 표시한다. (HITL 재실행에서도 공용)"""
        q = st.session_state.last_question
        with st.status("ReAct 루프 실행 중...", expanded=True) as status:
            steps, ans = core.run_react_loop(
                get_llm(model_name, 0.0, 512), q, lang=wiki_lang, max_steps=max_steps,
                enable_web=enable_web, engines=selected_engines, rag_chunks=rag_chunks,
                preset_history=preset_history, on_step=lambda s: render_step(s, status),
            )
            status.update(label=f"ReAct 완료 ({len(steps)} steps)", state="complete", expanded=False)
        st.session_state.update(react_steps=steps, react_answer=ans, trace_mode="react",
                                run_id=st.session_state.run_id + 1)

    if st.button("에이전트 실행", type="primary"):
        if not question.strip():
            st.warning("질문을 입력하세요.")
            st.stop()
        st.session_state.update(last_question=question.strip(), reflexion_trials=[], reflexion_memory=[])
        try:
            ensure_ollama_model(model_name)
            llm = get_llm(model_name, 0.0, 512)

            if mode.startswith("CoT"):
                with st.spinner("CoT (Reason Only) 실행 중..."):
                    resp = get_llm(model_name, 0.0, 1024).invoke(
                        f"Question: {question}\nLet's think step by step using only internal knowledge."
                    ).content
                st.session_state.update(
                    react_steps=[{"step": 1, "thought": resp, "action": "finish[CoT]", "observation": "외부 검색 미사용"}],
                    react_answer=resp, trace_mode="cot",
                )
            elif mode.startswith("Reflexion"):
                with st.status(f"Reflexion 루프 실행 중 (최대 {max_trials} Trials, Ω={memory_capacity})...", expanded=True) as status:
                    trials, ans, mem = core.run_reflexion_loop(
                        llm=llm, question=question, lang=wiki_lang, max_steps=max_steps,
                        max_trials=max_trials, memory_capacity=memory_capacity, enable_web=enable_web,
                        engines=selected_engines, rag_chunks=rag_chunks,
                        on_step=lambda s: render_step(s, status),
                        on_trial=lambda n: status.markdown(f"#### Trial {n}"),
                        eval_llm=None if eval_model_name == model_name else get_llm(eval_model_name, 0.0, 512),
                    )
                    status.update(label=f"Reflexion 완료 ({len(trials)} trial)", state="complete", expanded=False)
                st.session_state.update(
                    reflexion_trials=trials, reflexion_memory=mem,
                    react_steps=trials[-1]["steps"] if trials else [], react_answer=ans, trace_mode="reflexion",
                )
            elif mode.startswith("LangGraph"):
                with st.spinner("LangGraph ReAct Agent (RAG + 인터넷 검색) 실행 중..."):
                    lg_sentences: list[str] = []
                    lg_title = [""]

                    @tool
                    def search_wiki(query: str) -> str:
                        """Open the Wikipedia page of one entity (person, work, place). Pass a short entity title."""
                        obs, new_sents = core.wiki_search_api(query, lang=wiki_lang, context=question)
                        if new_sents:
                            lg_sentences[:] = new_sents
                            lg_title[0] = core._page_title(obs)
                        return obs

                    @tool
                    def lookup_wiki(keyword: str) -> str:
                        """Find sentences containing a keyword in the Wikipedia page opened by search_wiki."""
                        return core.wiki_lookup_api(keyword, lg_sentences, page_title=lg_title[0])

                    @tool
                    def calculator(expression: str) -> str:
                        """Evaluate a pure arithmetic expression such as '2884 + 220'."""
                        try:
                            return core.safe_calc(expression)
                        except Exception as exc:
                            return f"Calc error: {exc}"

                    lg_tools = [search_wiki, lookup_wiki, calculator]
                    if rag_chunks:
                        @tool
                        def rag_retriever(query: str) -> str:
                            """Search the user's uploaded local document. Use first for questions about the uploaded file."""
                            return core.simple_rag_search(query, rag_chunks)
                        lg_tools.insert(0, rag_retriever)
                    if enable_web:
                        @tool
                        def web_search(query: str) -> str:
                            """Search the live internet. Call this when Wikipedia or the local document lacks the fact."""
                            return core.web_search_api(query, engines=selected_engines)
                        lg_tools.append(web_search)

                    lg_sys = (
                        "You are a strictly grounded ReAct research agent. Never answer from memory: "
                        "verify every fact with a tool call first. If a tool returns an error, switch to another tool "
                        "or query instead of repeating it. Respond in Korean."
                    )
                    agent = create_modern_agent(model=llm, tools=lg_tools)
                    try:
                        res = agent.invoke(
                            {"messages": [SystemMessage(content=lg_sys), HumanMessage(content=question)]},
                            config={"recursion_limit": 2 * max_steps + 3},
                        )
                        messages = res["messages"]
                        answer = str(messages[-1].content)
                    except Exception as exc:
                        if "recursion" not in type(exc).__name__.lower():
                            raise
                        messages, answer = [], core.MAX_STEP_FAIL_MSG

                    trace = []
                    for msg in messages:
                        if msg.type in ("system", "human"):
                            continue
                        calls = getattr(msg, "tool_calls", None) or []
                        if calls:
                            action = " ; ".join(f"{c['name']}[{c['args']}]" for c in calls)
                        elif msg.type == "tool":
                            action = f"(tool result) {getattr(msg, 'name', '')}"
                        else:
                            action = "finish"
                        trace.append({"step": len(trace) + 1, "thought": f"[{msg.type.upper()}]",
                                      "action": action, "observation": str(msg.content)[:700]})
                    st.session_state.update(react_steps=trace, react_answer=answer, trace_mode="langgraph")
            else:
                run_react()
        except Exception as exc:
            st.error(f"에이전트 실행 오류: {type(exc).__name__}: {exc}")

    if st.session_state.reflexion_trials:
        st.subheader("최종 답변 (Reflexion)")
        st.info(st.session_state.react_answer)

        if st.session_state.reflexion_memory:
            st.subheader("Episodic Memory Buffer (장기 기억 · Self-Reflections)")
            for m_idx, mem_item in enumerate(st.session_state.reflexion_memory, start=1):
                st.warning(f"Memory #{m_idx}: {mem_item}")

        st.subheader("Reflexion Trial 별 Actor · Evaluator · Self-Reflection 궤적")
        last_trial = st.session_state.reflexion_trials[-1]["trial"]
        for tr in st.session_state.reflexion_trials:
            ev = tr["evaluation"]
            badge = "✅ PASS (r=1)" if ev["passed"] else "❌ FAIL (r=0)"
            with st.expander(f"Trial #{tr['trial']} — {badge} | 후보 답변: {tr['answer'][:80]}", expanded=tr["trial"] == last_trial):
                st.markdown(f"**Evaluator (M_e) 판정:** {badge} — {ev['reason']}")
                if tr.get("reflection"):
                    st.markdown(f"**Self-Reflection (M_sr) 피드백:** {tr['reflection']}")
                for s in tr["steps"]:
                    render_step(s)
    elif st.session_state.react_steps:
        st.subheader("최종 답변")
        st.info(st.session_state.react_answer)

        hitl = st.session_state.trace_mode == "react"
        st.subheader("Thought · Action · Observation 실행 궤적" + (" & HITL 사고 편집" if hitl else ""))
        if hitl:
            st.caption("Thought를 고친 뒤 ‘이 Thought로 재실행’을 누르면, 그 스텝부터 수정된 사고를 따라 다시 추론합니다.")
        steps = st.session_state.react_steps
        rerun_from = None
        for idx, s in enumerate(steps):
            with st.expander(f"Step {s['step']} — {s.get('action', '')[:90]}", expanded=True):
                render_step(s)
                if hitl and s["step"] <= max_steps and not s.get("action", "").startswith("finish["):
                    edited = st.text_area(
                        f"Thought {s['step']} 편집", value=s.get("thought", ""), height=70,
                        key=f"hitl_{st.session_state.run_id}_{idx}",
                    )
                    if st.button("이 Thought로 재실행", key=f"hitl_btn_{st.session_state.run_id}_{idx}") and edited.strip():
                        rerun_from = (idx, edited.strip())
        if rerun_from is not None:
            idx, edited = rerun_from
            try:
                run_react(preset_history=steps[:idx] + [{"step": steps[idx]["step"], "thought": edited}])
                st.rerun()
            except Exception as exc:
                st.error(f"재실행 오류: {type(exc).__name__}: {exc}")

# ---------------------------------------------------------
# 4. Chat tab
# ---------------------------------------------------------
with tab_chat:
    st.subheader("💬 OLLAMA 8B와 자유롭게 나누는 다이렉트 챗")
    st.caption("에이전트 제약 없이 로컬 LLM 모델과 자유 대화를 나누거나, 업로드된 문서 내용을 참고하도록 할 수 있습니다.")
    c1, c2 = st.columns([3, 1])
    chat_use_web = c1.toggle("답변 전에 실시간 웹 검색 컨텍스트 붙이기", value=enable_web, disabled=not enable_web,
                             help="끄면 검색 없이 바로 답해 훨씬 빠릅니다.")
    if c2.button("대화 초기화"):
        st.session_state.chat_history = []

    for msg in st.session_state.chat_history:
        with st.chat_message(msg["role"]):
            st.write(msg["content"])

    if user_prompt := st.chat_input("Ollama 8B 모델에게 무엇이든 물어보세요..."):
        with st.chat_message("user"):
            st.write(user_prompt)

        with st.chat_message("assistant"):
            try:
                ensure_ollama_model(model_name)
                system_msg = "당신은 사용자와 자유롭게 대화하는 성실한 AI 어시스턴트입니다. 한국어 질문에는 한국어로 답하세요.\n"
                if chat_use_web:
                    with st.spinner("최신 검색 중..."):
                        web_ctx = core.web_search_api(user_prompt, engines=selected_engines, max_per_engine=2)
                    if web_ctx != core.NO_WEB_MSG:
                        system_msg += (
                            "\n[중요 지침]\n"
                            "- 최신 정보나 확실히 알지 못하는 사실은 아래 검색 결과에 근거해서만 답하고, 근거가 없으면 모른다고 답하세요.\n"
                            "- 지어낸 정보를 절대 포함하지 마세요.\n\n"
                            f"[실시간 인터넷 검색 결과]:\n{web_ctx}\n"
                        )
                if rag_chunks:
                    system_msg += f"\n[사용자 업로드 로컬 문서 발췌]:\n{core.simple_rag_search(user_prompt, rag_chunks, top_k=3)}\n"

                chat_messages = [SystemMessage(content=system_msg)]
                for h in st.session_state.chat_history[-6:]:      # 이전 턴은 역할을 구분해서 전달
                    chat_messages.append((HumanMessage if h["role"] == "user" else AIMessage)(content=h["content"]))
                chat_messages.append(HumanMessage(content=user_prompt))

                chat_llm = get_llm(model_name, 0.3, 1024)
                response = st.write_stream(chunk.content for chunk in chat_llm.stream(chat_messages))
                st.session_state.chat_history.append({"role": "user", "content": user_prompt})
                st.session_state.chat_history.append({"role": "assistant", "content": response})
            except Exception as e:
                st.error(f"Ollama 호출 오류: {e}")


In [ ]:
import os
import re
import shutil
import subprocess
import time
import requests

# 1) cloudflared 설치 (이미 있으면 건너뜀)
if shutil.which("cloudflared") is None:
    subprocess.run("curl -fsSL -o cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared.deb",
                   shell=True, check=True, stdout=subprocess.DEVNULL)

# 2) 이전에 띄운 프로세스 정리 (셀을 다시 실행해도 포트 충돌·옛 코드 잔존이 없도록)
subprocess.run(["pkill", "-f", "streamlit run app.py"], check=False)
subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
time.sleep(1)
for f in ("cloudflared.log", "streamlit.log"):
    if os.path.exists(f):
        os.remove(f)        # 옛 로그의 죽은 URL을 읽지 않도록 삭제

# 3) Streamlit 백그라운드 실행 (터널 프록시 뒤에서 파일 업로드가 403 나지 않도록 CORS/XSRF 해제)
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
     "--browser.gatherUsageStats", "false"],
    stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(40):
    try:
        if requests.get("http://localhost:8501/_stcore/health", timeout=1).ok:
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Streamlit이 시작되지 않았습니다:\n" + open("streamlit.log").read()[-1500:])
print("✅ Streamlit 정상 가동 (localhost:8501)")

# 4) Cloudflare Quick Tunnel 실행 후 URL이 나올 때까지 대기 (최대 40초)
subprocess.Popen("cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1", shell=True)
print("⏳ Cloudflare Tunnel URL 생성 대기 중...")
urls = []
for _ in range(40):
    if os.path.exists("cloudflared.log"):
        urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
        if urls:
            break
    time.sleep(1)

if urls:
    print(f"\n✅ Streamlit 접속 주소: {urls[-1]}")
    print("   (주소가 열리지 않으면 DNS 전파까지 10~20초 기다렸다가 새로고침하세요)")
else:
    print("\n★ 터널 URL을 받지 못했습니다. 로그:")
    print(open("cloudflared.log").read()[-800:] if os.path.exists("cloudflared.log") else "(cloudflared.log 없음)")